In [ ]:
from tensorflow.keras.models import Model,load_model
from tensorflow.keras.applications import EfficientNetV2B3
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from sklearn.metrics import classification_report,confusion_matrix
import clahe                                              

I0000 00:00:1793937022.265928    9837 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793937024.012630    9837 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793937034.574870    9837 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
train_path = "../Datasets/data/train/Medical/X-Ray/forearm/Train"
valid_path = "../Datasets/data/vaild/Medical/X-Ray/forearm"
testing_path = "../Datasets/data/test/Medical/X-Ray/forearm/Test"

In [3]:
train_aug = ImageDataGenerator(
    preprocessing_function = lambda img : clahe.clahe_preprocess(img,preprocess_input),
    rotation_range = 10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    shear_range=0.05,
)
valid_aug = ImageDataGenerator(
    preprocessing_function=lambda img: clahe.clahe_preprocess(img, preprocess_input),
)
test_aug = ImageDataGenerator(preprocessing_function = lambda img : clahe.clahe_preprocess(img,preprocess_input))

In [4]:
train_set = train_aug.flow_from_directory(
    train_path,
    target_size = (256,256),
    batch_size=8,
    class_mode="binary",
    shuffle = False

)
validation_set = train_aug.flow_from_directory(
    valid_path,
    target_size=(256, 256),
    batch_size=8,
    class_mode="binary",
)
test_set = test_aug.flow_from_directory(
    testing_path,
    target_size = (256,256),
    batch_size=8,
    class_mode="binary",
    shuffle = False
)

Found 2502 images belonging to 2 classes.
Found 500 images belonging to 2 classes.
Found 767 images belonging to 2 classes.


In [5]:
base_model = EfficientNetV2B3(include_top=False,weights="imagenet",input_shape=(256,256,3),pooling='avg')

I0000 00:00:1793937050.104947    9837 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


In [6]:
base_model.trainable = False

In [7]:
x = base_model.output
x = Dense(64,activation='relu')(x)
output = Dense(1,activation='sigmoid')(x)

In [8]:
model = Model(inputs = base_model.input,outputs = output)

In [9]:
model.compile(optimizer= Adam(learning_rate=0.00028),loss="binary_crossentropy",metrics=['accuracy'])

In [10]:
erly_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

In [11]:
class_weight = {0:2,1:1.1}

In [12]:
model.fit(train_set,validation_data=validation_set,epochs=40,callbacks=erly_stop)

Epoch 1/40


I0000 00:00:1793937061.289157    9837 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1793937073.069745   10082 service.cc:153] XLA service 0x7a519c04dbc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1793937073.069817   10082 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2050, Compute Capability 8.6 (Driver: 13.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1793937073.703089   10082 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1793937077.076597   10082 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1793937078.228500   10082 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_27348__.389
I0000 00:00:1793937101.848700   10082 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


249/313 ━━━━━━━━━━━━━━━━━━━━ 7s 115ms/step - accuracy: 0.6800 - loss: 0.6150

I0000 00:00:1793937132.247153   10080 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_27348__.389


313/313 ━━━━━━━━━━━━━━━━━━━━ 124s 265ms/step - accuracy: 0.7258 - loss: 0.5478 - val_accuracy: 0.7300 - val_loss: 0.5918
Epoch 2/40
313/313 ━━━━━━━━━━━━━━━━━━━━ 67s 126ms/step - accuracy: 0.7930 - loss: 0.4642 - val_accuracy: 0.7620 - val_loss: 0.5067
Epoch 3/40
313/313 ━━━━━━━━━━━━━━━━━━━━ 38s 120ms/step - accuracy: 0.8149 - loss: 0.4159 - val_accuracy: 0.7820 - val_loss: 0.4785
Epoch 4/40
313/313 ━━━━━━━━━━━━━━━━━━━━ 38s 120ms/step - accuracy: 0.8118 - loss: 0.4210 - val_accuracy: 0.7400 - val_loss: 0.5581
Epoch 5/40
313/313 ━━━━━━━━━━━━━━━━━━━━ 34s 109ms/step - accuracy: 0.8157 - loss: 0.4169 - val_accuracy: 0.7920 - val_loss: 0.4730
Epoch 6/40
313/313 ━━━━━━━━━━━━━━━━━━━━ 39s 123ms/step - accuracy: 0.8357 - loss: 0.3837 - val_accuracy: 0.7980 - val_loss: 0.4525
Epoch 7/40
313/313 ━━━━━━━━━━━━━━━━━━━━ 35s 110ms/step - accuracy: 0.8433 - loss: 0.3643 - val_accuracy: 0.7980 - val_loss: 0.4693
Epoch 8/40
313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 114ms/step - accuracy: 0.8465 - loss: 0.3588 - va

In [13]:
# model_path = "../models/foreaam_x_ray_model.keras"

# model = load_model(model_path)

In [14]:
prob = model.predict(test_set)

96/96 ━━━━━━━━━━━━━━━━━━━━ 24s 178ms/step


In [15]:
pred = (prob > 0.5).astype(int)

In [16]:
true_data = test_set.classes
model1 = model

In [17]:
print(classification_report(true_data,pred))
print(confusion_matrix(true_data,pred))

              precision    recall  f1-score   support

           0       0.91      0.74      0.82       385
           1       0.78      0.92      0.85       382

    accuracy                           0.83       767
   macro avg       0.84      0.83      0.83       767
weighted avg       0.84      0.83      0.83       767

[[286  99]
 [ 29 353]]


In [18]:
train_true = train_set.classes
train_pred = model.predict(train_set)

313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 114ms/step


In [19]:
train_pred = (train_pred > 0.5).astype(int)

In [20]:
print(classification_report(train_true,train_pred))
print(confusion_matrix(train_true,train_pred))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95      1074
           1       0.96      0.96      0.96      1428

    accuracy                           0.96      2502
   macro avg       0.96      0.96      0.96      2502
weighted avg       0.96      0.96      0.96      2502

[[1018   56]
 [  50 1378]]


In [21]:
base_model.trainable = True

In [22]:
for layer in base_model.layers[:-30]:
    layer.trainable = False

In [23]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2
)

In [24]:
model.compile(optimizer=Adam(learning_rate = 0.0005),loss="binary_crossentropy",metrics=['accuracy'])

In [25]:
model.fit(train_set,validation_data=validation_set,epochs=30,callbacks=[reduce_lr,early_stop])

Epoch 1/30


I0000 00:00:1793938286.457881   10081 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_132412__.413


197/313 ━━━━━━━━━━━━━━━━━━━━ 9s 86ms/step - accuracy: 0.7130 - loss: 0.5869 

I0000 00:00:1793938324.989822   10082 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_132412__.413


313/313 ━━━━━━━━━━━━━━━━━━━━ 105s 218ms/step - accuracy: 0.8054 - loss: 0.4314 - val_accuracy: 0.7680 - val_loss: 0.5144 - learning_rate: 5.0000e-04
Epoch 2/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 37s 116ms/step - accuracy: 0.9317 - loss: 0.1845 - val_accuracy: 0.5340 - val_loss: 1.2805 - learning_rate: 5.0000e-04
Epoch 3/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 105ms/step - accuracy: 0.9600 - loss: 0.1194 - val_accuracy: 0.6460 - val_loss: 0.7230 - learning_rate: 5.0000e-04
Epoch 4/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 37s 118ms/step - accuracy: 0.9892 - loss: 0.0476 - val_accuracy: 0.7640 - val_loss: 0.4731 - learning_rate: 1.0000e-04
Epoch 5/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 35s 110ms/step - accuracy: 0.9944 - loss: 0.0310 - val_accuracy: 0.6460 - val_loss: 0.6777 - learning_rate: 1.0000e-04
Epoch 6/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 35s 113ms/step - accuracy: 0.9916 - loss: 0.0310 - val_accuracy: 0.6740 - val_loss: 0.6613 - learning_rate: 1.0000e-04
Epoch 7/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 37s 117ms/step -

In [26]:
prob = model.predict(test_set)

pred = (prob >= 0.5).astype(int)
true_data = test_set.classes
model1 = model

print(classification_report(true_data,pred))
print(confusion_matrix(true_data,pred))

96/96 ━━━━━━━━━━━━━━━━━━━━ 18s 112ms/step
              precision    recall  f1-score   support

           0       0.76      0.67      0.71       385
           1       0.70      0.79      0.75       382

    accuracy                           0.73       767
   macro avg       0.73      0.73      0.73       767
weighted avg       0.73      0.73      0.73       767

[[257 128]
 [ 79 303]]


In [27]:
model1.save('foream_x_ray_model.keras')